In [ ]:
import os
import shutil
import sys
from datetime import date
from pathlib import Path

from hydra import compose, initialize_config_dir

import peract_config

In [ ]:
root = Path("../..").resolve()
artifact_root = root / "own_code/artifacts"
run_date = date.today().isoformat()
dataset_path = artifact_root / "datasets/peract2"
checkpoint_dir = artifact_root / "checkpoints/peract2" / run_date
run_dir = artifact_root / "runs/peract2" / run_date
sim = root / "sim/CoppeliaSim_Edu_V4_1_0_Ubuntu20_04"
peract_dir = root / "peract_bimanual"
os.environ["COPPELIASIM_ROOT"] = str(sim)
os.environ["LD_LIBRARY_PATH"] = f"{os.environ.get('LD_LIBRARY_PATH', '')}:{sim}"
os.environ["QT_QPA_PLATFORM_PLUGIN_PATH"] = str(sim)
sys.path.insert(0, str(peract_dir))

In [ ]:
from train import run_training

with initialize_config_dir(version_base="1.1", config_dir=str(peract_dir / "conf")):
    config = compose(config_name="config", overrides=["method=BIMANUAL_PERACT"])

config.rlbench.tasks = ["bimanual_dual_push_buttons"]
config.rlbench.task_name = "bimanual_dual_push_buttons"
config.rlbench.demos = 1
config.rlbench.demo_path = str(dataset_path)

config.replay.batch_size = 1
config.replay.capacity = 100
config.replay.use_disk = False

config.framework.training_iterations = 1
config.framework.save_freq = 1
config.framework.load_existing_weights = False

config.method.voxel_sizes = [20]
config.method.num_latents = 64
config.method.latent_dim = 64
config.method.transformer_depth = 1
config.method.latent_heads = 4
config.method.latent_dim_head = 16

In [ ]:
peract_config.on_init()
run_training(config, run_dir)

In [ ]:
weights_dir = run_dir / "bimanual_dual_push_buttons/BIMANUAL_PERACT/seed0/weights"
latest_weight = max(weights_dir.iterdir(), key=lambda path: int(path.name))
latest_dir = checkpoint_dir / "latest"
if latest_dir.exists():
    shutil.rmtree(latest_dir)
shutil.move(latest_weight, latest_dir)